# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. This dataset contains multiple record sets, fields, and columns as defined by its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dot-accessible object, not a dict
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}\nAuthors: {metadata.author}")

## 2. Data Overview

Review available record sets and their fields, referencing everything by their `@id`.

Below, we display all available record sets in the dataset, the fields/columns each exposes, and the field `@id` values to ease referencing.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)  # Each is a mlcroissant.RecordSet object
print(f"Total record sets found: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (with @id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}]")
    print("  Columns (with @id):")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print('-'*60)
print("All record set @id values:")
for idx, rid in enumerate(record_set_ids):
    print(f"  {idx}. {rid}")

## 3. Data Extraction

Load data from the main record set(s) into a DataFrame for analysis. We use the record set and field `@id` values as referenced above.

**Note:** If your dataset contains multiple record sets, they will each be loaded separately into individual DataFrames.

In [ ]:
# Choose record sets for extraction
record_set_ids  # This was built above

# We will load all record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    # Each record set may have hundreds/thousands of records. Below will load all.
    print(f"Extracting records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    dataframes[rs_id] = df
    print(df.head(2))
    print('-'*40)
# Pick the main table for analysis below. Usually, the most substantial record set holds the proteins summary; adjust @id as necessary.
main_record_set_id = record_set_ids[0] if record_set_ids else None  # Example; update with the relevant @id
main_df = dataframes[main_record_set_id]
print(f"Columns in main DataFrame ({main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records by numeric values, normalizing columns, grouping, and examining characteristics.

Reference all fields by their `@id` as in previous cells.

In [ ]:
# Example: Analyze a numeric field.
# List numeric fields (fields with data_type float, number, or integer)
main_rs = next((rs for rs in dataset.record_sets if rs.id == main_record_set_id), None)
numeric_fields = [f for f in main_rs.fields if (f.data_type is not None and f.data_type.lower() in ['float', 'number', 'integer'])]
print("Numeric fields and their @id:")
for f in numeric_fields:
    print(f"  - {f.name} (@id: {f.id})")

# Let's pick the first available numeric field for demonstration, or set manually if known
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    numeric_field_name = numeric_fields[0].name
else:
    numeric_field_id = main_df.columns[0]  # fallback
    numeric_field_name = main_df.columns[0]
print(f"Using numeric field: {numeric_field_name} (@id: {numeric_field_id})")

# Clean and convert to numeric (set errors to NaN)
col = numeric_field_id
main_df[col] = pd.to_numeric(main_df[col], errors='coerce')

threshold = main_df[col].mean()  # As an example threshold
filtered_df = main_df[main_df[col] > threshold]
print(f"There are {len(filtered_df)} records with {col} > mean ({threshold:.2f})")
print(filtered_df[[col]].head())

# Normalization
filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
print(filtered_df[[col, f"{col}_normalized"]].head())

# Try to group by a categorical field, e.g. protein description or sample (search for a field with data_type == 'Text')
cat_fields = [f for f in main_rs.fields if f.data_type and f.data_type.lower() == 'text']
if cat_fields:
    group_field_id = cat_fields[0].id
    print(f"Grouping by field: {cat_fields[0].name} (@id: {group_field_id})")
    grouped = filtered_df.groupby(group_field_id)[col].mean()
    print(grouped.head())

## 5. Visualization

Visualize data distributions or relationships between selected fields in the dataset. Below is an example histogram and boxplot for the analyzed numeric field, colored by a categorical variable (when available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(data=main_df, x=numeric_field_id, bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_name} (@id: {numeric_field_id})")
plt.xlabel(numeric_field_name)
plt.show()

if cat_fields:
    # Boxplot by group
    plt.figure(figsize=(12,4))
    sns.boxplot(
        data=filtered_df,
        x=group_field_id,
        y=numeric_field_id
    )
    plt.title(f"{numeric_field_name} by {cat_fields[0].name}")
    plt.ylabel(numeric_field_name)
    plt.xlabel(cat_fields[0].name)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We examined available record sets and fields (referenced by their `@id`s), performed basic extraction of the main table, filtered and normalized a numeric field, and visualized the data distribution.

**Key findings and next steps:**
- The dataset exposes detailed protein measurements suitable for downstream analysis.
- The structure, via record sets and field IDs, makes processing extensible for larger workflows.
- Use the available field `@id` references to ensure reproducibility in scripts and collaborative projects.

Further analysis can include:
- Deeper data cleaning and missing value analysis
- Multi-field visualization and feature engineering
- Statistical testing between sample groups or experimental conditions
